# Arquitectura Medallion Polyglot (Parte 3 - Capa Gold, SQL y Power BI)


## PASO 0: Inicio de sesión Spark con Hive


In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("iberian_logistics_gold")
    .config("spark.sql.shuffle.partitions", "4")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

BASE_PATH = "/opt/datos/iberian_logistics_polyglot"
GOLD_PATH = f"{BASE_PATH}/gold"
PBI_CSV_PATH = f"{BASE_PATH}/powerbi_csv"

Path(GOLD_PATH).mkdir(parents=True, exist_ok=True)
Path(PBI_CSV_PATH).mkdir(parents=True, exist_ok=True)

spark.sql("USE iberian_logistics_db")

print("Base de datos activa:", spark.catalog.currentDatabase())
spark.sql("SHOW TABLES").show(truncate=False)

NOTE: Picked up JDK_JAVA_OPTIONS: -Duser.home=/home/spark -Dlog4j2.configurationFile=/opt/spark/conf/log4j2.properties --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/jdk.internal.loader=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=tru

Base de datos activa: iberian_logistics_db
+--------------------+-----------------------+-----------+
|namespace           |tableName              |isTemporary|
+--------------------+-----------------------+-----------+
|iberian_logistics_db|bronze_customers       |false      |
|iberian_logistics_db|bronze_orders_shipments|false      |
|iberian_logistics_db|silver_customers       |false      |
|iberian_logistics_db|silver_orders_shipments|false      |
+--------------------+-----------------------+-----------+



## Creación de los 3 Datamarts para Power BI
Lee las tablas limpias `silver_customers` y `silver_orders_shipments` y construye los 3 Datamarts de negocio en Hive.

In [2]:
df_cust_silver = spark.table("iberian_logistics_db.silver_customers")
df_ord_silver = spark.table("iberian_logistics_db.silver_orders_shipments")

# join entre pedidos y clientes en df_enriched
df_enriched = (
    df_ord_silver.alias("o")
    .join(df_cust_silver.alias("c"), on="customer_id", how="inner")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.full_name",
        "c.city",
        "c.segment",
        "c.age",
        "o.order_date",
        F.date_format("o.order_date", "yyyy-MM").alias("year_month"),
        "o.product_category",
        "o.quantity",
        "o.unit_price",
        "o.shipping_cost",
        "o.gross_amount",
        "o.total_order_value",
        "o.carrier",
        "o.delivery_status",
        "o.promised_delivery_days",
        "o.actual_delivery_days",
        "o.delay_days",
        "o.is_on_time"
    )
)

print("Filas en df_enriched:", df_enriched.count())

# Datamart 1 iberian_logistics_db.gold_dm_sales_performance
df_dm_sales = (
    df_enriched
    .filter(F.col("delivery_status") != "CANCELLED")
    .groupBy("year_month", "city", "segment", "product_category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("gross_amount"), 2).alias("total_gross_revenue"),
        F.round(F.sum("total_order_value"), 2).alias("total_billed_amount"),
        F.round(F.avg("total_order_value"), 2).alias("avg_order_ticket")
    )
    .orderBy("year_month", "city", "product_category")
)

(
    df_dm_sales.coalesce(1).write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{GOLD_PATH}/dm_sales_performance")
    .saveAsTable("iberian_logistics_db.gold_dm_sales_performance")
)

#Datamart 2 iberian_logistics_db.gold_dm_logistics_sla
df_dm_sla = (
    df_enriched
    .groupBy("carrier", "city", "delivery_status")
    .agg(
        F.countDistinct("order_id").alias("total_shipments"),
        F.round(F.avg("promised_delivery_days"), 2).alias("avg_promised_days"),
        F.round(F.avg("actual_delivery_days"), 2).alias("avg_actual_days"),
        F.round(F.avg("delay_days"), 2).alias("avg_delay_days"),
        F.round(
            F.avg(F.when(F.col("is_on_time") == True, 100.0).otherwise(0.0)),
            2
        ).alias("sla_on_time_pct"),
        F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost")
    )
    .orderBy("carrier", "city", "delivery_status")
)

(
    df_dm_sla.coalesce(1).write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{GOLD_PATH}/dm_logistics_sla")
    .saveAsTable("iberian_logistics_db.gold_dm_logistics_sla")
)

# Datamart 3iberian_logistics_db.gold_dm_customer_360_rfm
ref_date = df_enriched.agg(F.max("order_date")).collect()[0][0]

df_dm_rfm = (
    df_enriched
    .filter(F.col("delivery_status") != "CANCELLED")
    .groupBy("customer_id", "full_name", "city", "segment", "age")
    .agg(
        F.countDistinct("order_id").alias("frequency_orders"),
        F.round(F.sum("total_order_value"), 2).alias("monetary_ltv"),
        F.round(F.avg("total_order_value"), 2).alias("avg_ticket"),
        F.max("order_date").alias("last_order_date"),
        F.round(
            F.avg(F.when(F.col("is_on_time") == True, 100.0).otherwise(0.0)),
            2
        ).alias("customer_on_time_experience_pct")
    )
    .withColumn("recency_days", F.datediff(F.lit(ref_date), F.col("last_order_date")))
    .withColumn(
        "rfm_tier",
        F.when(
            (F.col("recency_days") <= 45) & (F.col("frequency_orders") >= 5) & (F.col("monetary_ltv") >= 5000),
            "Champion"
        )
        .when(
            (F.col("frequency_orders") >= 4) & (F.col("monetary_ltv") >= 4000),
            "Loyal High-Value"
        )
        .when(
            F.col("recency_days") > 90,
            "At Risk / Churn"
        )
        .otherwise("Regular")
    )
    .orderBy(F.desc("monetary_ltv"))
)

(
    df_dm_rfm.coalesce(1).write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{GOLD_PATH}/dm_customer_360_rfm")
    .saveAsTable("iberian_logistics_db.gold_dm_customer_360_rfm")
)

print("Datamarts Gold creados correctamente:")
print(" - gold_dm_sales_performance :", spark.table("iberian_logistics_db.gold_dm_sales_performance").count(), "filas")
print(" - gold_dm_logistics_sla     :", spark.table("iberian_logistics_db.gold_dm_logistics_sla").count(), "filas")
print(" - gold_dm_customer_360_rfm  :", spark.table("iberian_logistics_db.gold_dm_customer_360_rfm").count(), "filas")

Filas en df_enriched: 609


Datamarts Gold creados correctamente:
 - gold_dm_sales_performance : 361 filas
 - gold_dm_logistics_sla     : 110 filas
 - gold_dm_customer_360_rfm  : 129 filas


## RESOLUCIÓN DE LAS 4 PREGUNTAS DE NEGOCIO 


In [3]:
# Top 5 combinaciones Ciudad + Categoría por facturación total y ticket medio
q1 = spark.sql("""
    SELECT
        city,
        product_category,
        SUM(total_orders) AS total_orders,
        ROUND(SUM(total_billed_amount), 2) AS total_facturacion,
        ROUND(SUM(total_billed_amount) / SUM(total_orders), 2) AS ticket_medio
    FROM iberian_logistics_db.gold_dm_sales_performance
    GROUP BY city, product_category
    ORDER BY total_facturacion DESC
    LIMIT 5
""")
print("=== PREGUNTA 1: Top 5 Ciudad + Categoría ===")
q1.show(truncate=False)

=== PREGUNTA 1: Top 5 Ciudad + Categoría ===
+---------+----------------+------------+-----------------+------------+
|city     |product_category|total_orders|total_facturacion|ticket_medio|
+---------+----------------+------------+-----------------+------------+
|Bilbao   |Moda            |22          |33259.08         |1511.78     |
|Zaragoza |Electrónica     |18          |33002.19         |1833.46     |
|Barcelona|Electrónica     |17          |32312.99         |1900.76     |
|Bilbao   |Alimentación    |19          |31508.37         |1658.34     |
|Bilbao   |Deportes        |20          |27769.53         |1388.48     |
+---------+----------------+------------+-----------------+------------+



In [4]:
# Operador logístico con peor % de SLA en envíos entregados y retraso medio
q2 = spark.sql("""
    SELECT
        carrier,
        SUM(total_shipments) AS total_delivered,
        ROUND(SUM(sla_on_time_pct * total_shipments) / SUM(total_shipments), 2) AS sla_on_time_pct,
        ROUND(SUM(avg_delay_days * total_shipments) / SUM(total_shipments), 2) AS retraso_medio_dias
    FROM iberian_logistics_db.gold_dm_logistics_sla
    WHERE delivery_status = 'DELIVERED'
    GROUP BY carrier
    ORDER BY sla_on_time_pct ASC
""")
print("=== PREGUNTA 2: SLA On-Time y retraso medio por Carrier (DELIVERED) ===")
q2.show(truncate=False)

=== PREGUNTA 2: SLA On-Time y retraso medio por Carrier (DELIVERED) ===
+-------------+---------------+---------------+------------------+
|carrier      |total_delivered|sla_on_time_pct|retraso_medio_dias|
+-------------+---------------+---------------+------------------+
|TransNorte   |82             |57.32          |0.93              |
|RápidoSur    |62             |62.9           |0.39              |
|EcoLogix     |56             |64.29          |0.84              |
|ExpressIberia|59             |66.1           |0.68              |
+-------------+---------------+---------------+------------------+



In [5]:
#Clientes por segmento rfm_tier y LTV acumulado
q3 = spark.sql("""
    SELECT
        rfm_tier,
        COUNT(DISTINCT customer_id) AS num_clientes,
        ROUND(SUM(monetary_ltv), 2) AS ltv_acumulado,
        ROUND(AVG(monetary_ltv), 2) AS ltv_medio
    FROM iberian_logistics_db.gold_dm_customer_360_rfm
    GROUP BY rfm_tier
    ORDER BY ltv_acumulado DESC
""")
print("=== PREGUNTA 3: Clientes y LTV acumulado por RFM Tier ===")
q3.show(truncate=False)

=== PREGUNTA 3: Clientes y LTV acumulado por RFM Tier ===
+----------------+------------+-------------+---------+
|rfm_tier        |num_clientes|ltv_acumulado|ltv_medio|
+----------------+------------+-------------+---------+
|Champion        |24          |208398.23    |8683.26  |
|Loyal High-Value|26          |194359.17    |7475.35  |
|Regular         |48          |155367.21    |3236.82  |
|At Risk / Churn |31          |79262.29     |2556.85  |
+----------------+------------+-------------+---------+



In [6]:
#Porcentaje de pedidos de Bronze descartados al pasar a Silver
q4 = spark.sql("""
    WITH conteos AS (
        SELECT
            (SELECT COUNT(*) FROM iberian_logistics_db.bronze_orders_shipments) AS bronze_orders,
            (SELECT COUNT(*) FROM iberian_logistics_db.silver_orders_shipments) AS silver_orders
    )
    SELECT
        bronze_orders,
        silver_orders,
        (bronze_orders - silver_orders) AS pedidos_descartados,
        ROUND(((bronze_orders - silver_orders) * 100.0) / bronze_orders, 2) AS pct_descartados
    FROM conteos
""")
print("=== PREGUNTA 4: Porcentaje de pedidos descartados (Bronze -> Silver) ===")
q4.show(truncate=False)

=== PREGUNTA 4: Porcentaje de pedidos descartados (Bronze -> Silver) ===
+-------------+-------------+-------------------+---------------+
|bronze_orders|silver_orders|pedidos_descartados|pct_descartados|
+-------------+-------------+-------------------+---------------+
|853          |609          |244                |28.60          |
+-------------+-------------+-------------------+---------------+



## Exportación de los 3 Datamarts a CSV para Power BI Desktop
Guarda los 3 archivos listos para importar directamente en Power BI desde `C:\cursos\spark-cluster\datos\iberian_logistics_polyglot\powerbi_csv`.

In [7]:
tables_to_export = [
    "gold_dm_sales_performance",
    "gold_dm_logistics_sla",
    "gold_dm_customer_360_rfm"
]

for tbl in tables_to_export:
    pdf = spark.table(f"iberian_logistics_db.{tbl}").toPandas()
    out_file = f"{PBI_CSV_PATH}/{tbl}.csv"
    pdf.to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"Exportado: {out_file} ({len(pdf)} filas)")

print("\nEn tu Windows los encontrarás en:")
print(r"C:\cursos\spark-cluster\datos\iberian_logistics_polyglot\powerbi_csv")

Exportado: /opt/datos/iberian_logistics_polyglot/powerbi_csv/gold_dm_sales_performance.csv (361 filas)
Exportado: /opt/datos/iberian_logistics_polyglot/powerbi_csv/gold_dm_logistics_sla.csv (110 filas)
Exportado: /opt/datos/iberian_logistics_polyglot/powerbi_csv/gold_dm_customer_360_rfm.csv (129 filas)

En tu Windows los encontrarás en:
C:\cursos\spark-cluster\datos\iberian_logistics_polyglot\powerbi_csv


## Cerrar sesión


In [8]:
spark.stop()
print("Sesión de Spark cerrada correctamente.")

Sesión de Spark cerrada correctamente.
